# 🤖 Entrenamiento del Modelo y Predicción de Llaves - Mundial 2026

## 1. Carga de Variables Sintetizadas
En este cuaderno utilizaremos la matriz de características consolidada (`features_rendimiento_2026.csv`) para entrenar un modelo predictivo basado en Machine Learning (`RandomForestClassifier`) y simular los cruces restantes del torneo.

In [38]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# 1. Cargar nuestra matriz de características procesada
ruta_features = os.path.join('..', 'data', 'processed', 'features_rendimiento_2026.csv')
df_features = pd.read_csv(ruta_features)

print(f"✅ Matriz de entrenamiento cargada con éxito. Total de selecciones mapeadas: {len(df_features)}")
df_features.head()

✅ Matriz de entrenamiento cargada con éxito. Total de selecciones mapeadas: 48


,equipo,puntos_2026,gf_2026,gc_2026,pj_2026,dg_2026,partidos_jugados,victorias,efectividad_historica_%,puntos_por_pj_2026
0,France,12,13,2,4,11,77,43,55.84,3.0
1,Mexico,12,8,0,4,8,64,21,32.81,3.0
2,Brazil,10,9,2,4,7,118,79,66.95,2.5
3,England,10,8,3,4,5,78,35,44.87,2.5
4,Argentina,9,8,1,3,7,91,50,54.95,3.0


## 2. Definición del Cuadro de Eliminación Directa (Dieciseisavos de Final)
En esta sección estructuraremos el estado real del torneo en la fase de dieciseisavos. Registraremos tanto los partidos con resultados oficiales definitivos como las llaves que quedan pendientes por disputar. El modelo utilizará las métricas consolidadas para predecir el ganador de los encuentros que aún no se han jugado.

In [39]:
# 1. Mapeo de los 16 partidos de la ronda de Dieciseisavos de Final (Mundial 2026)
dieciseisavos_fixtures = [
    {"partido_id": 73, "home_team": "South Africa", "away_team": "Canada", "home_score": 0, "away_score": 1, "played": True},
    {"partido_id": 74, "home_team": "Brazil", "away_team": "Japan", "home_score": 2, "away_score": 1, "played": True},
    {"partido_id": 75, "home_team": "Germany", "away_team": "Paraguay", "home_score": 1, "away_score": 1, "ganador_penales": "Paraguay", "played": True},
    {"partido_id": 76, "home_team": "Netherlands", "away_team": "Morocco", "home_score": 1, "away_score": 1, "ganador_penales": "Morocco", "played": True},
    {"partido_id": 77, "home_team": "Ivory Coast", "away_team": "Norway", "home_score": 1, "away_score": 2, "played": True},
    {"partido_id": 78, "home_team": "France", "away_team": "Sweden", "home_score": 3, "away_score": 0, "played": True},
    {"partido_id": 79, "home_team": "Mexico", "away_team": "Ecuador", "home_score": 2, "away_score": 0, "played": True},
    {"partido_id": 80, "home_team": "England", "away_team": "DR Congo", "home_score": 2, "away_score": 1, "played": True},
    {"partido_id": 81, "home_team": "Belgium", "away_team": "Senegal", "home_score": 3, "away_score": 2, "played": True},
    {"partido_id": 82, "home_team": "United States", "away_team": "Bosnia and Herzegovina", "home_score": 2, "away_score": 0, "played": True},
    
    # Partidos pendientes por jugarse/simular
    {"partido_id": 83, "home_team": "Portugal", "away_team": "Croatia", "home_score": None, "away_score": None, "played": False},
    {"partido_id": 84, "home_team": "Spain", "away_team": "Austria", "home_score": None, "away_score": None, "played": False},
    {"partido_id": 85, "home_team": "Switzerland", "away_team": "Algeria", "home_score": None, "away_score": None, "played": False},
    {"partido_id": 86, "home_team": "Argentina", "away_team": "Cape Verde", "home_score": None, "away_score": None, "played": False},
    {"partido_id": 87, "home_team": "Colombia", "away_team": "Ghana", "home_score": None, "away_score": None, "played": False},
    {"partido_id": 88, "home_team": "Australia", "away_team": "Egypt", "home_score": None, "away_score": None, "played": False}
]

df_cruces_32 = pd.DataFrame(dieciseisavos_fixtures)
print(f"📋 Estructura de llaves cargada. Partidos jugados: {len(df_cruces_32[df_cruces_32['played'] == True])} | Pendientes: {len(df_cruces_32[df_cruces_32['played'] == False])}")

📋 Estructura de llaves cargada. Partidos jugados: 10 | Pendientes: 6


## 3. Estructura de Llaves: Octavos de Final
En esta sección definiremos la estructura de los Octavos de Final del Mundial 2026. Mapearemos los cruces fijos con selecciones ya clasificadas y dejaremos los espacios dinámicos (`Ganador Partido XX`) que serán resueltos automáticamente por nuestro modelo predictivo.

In [40]:
# Mapeo de los 8 partidos de la ronda de Octavos de Final (Mundial 2026)
octavos_fixtures = [
    {"partido_id": 89, "home_team": "Paraguay", "away_team": "France", "date": "2026-07-04"},
    {"partido_id": 90, "home_team": "Canada", "away_team": "Morocco", "date": "2026-07-04"},
    {"partido_id": 91, "home_team": "Brazil", "away_team": "Norway", "date": "2026-07-05"},
    {"partido_id": 92, "home_team": "Mexico", "away_team": "England", "date": "2026-07-05"},
    
    # Llaves dinámicas sujetas a los resultados de Dieciseisavos (Partidos 83 al 88)
    {"partido_id": 93, "home_team": "Ganador Partido 83", "away_team": "Ganador Partido 84", "date": "2026-07-06"},
    {"partido_id": 94, "home_team": "United States", "away_team": "Belgium", "date": "2026-07-06"},
    {"partido_id": 95, "home_team": "Ganador Partido 86", "away_team": "Ganador Partido 88", "date": "2026-07-07"},
    {"partido_id": 96, "home_team": "Ganador Partido 85", "away_team": "Ganador Partido 87", "date": "2026-07-07"}
]

df_cruces_16 = pd.DataFrame(octavos_fixtures)
print(f"✅ Cuadro de Octavos de Final inicializado correctamente ({len(df_cruces_16)} partidos).")

✅ Cuadro de Octavos de Final inicializado correctamente (8 partidos).


## 4. Estructura de Llaves Avanzadas: Cuartos de Final hasta la Final
En esta sección definimos las plantillas de los cruces de Cuartos de Final (Partidos 97-100), Semifinales (Partidos 101-102), Tercer Puesto (Partido 103) y la Gran Final (Partido 104). Estas llaves se resolverán en cascada utilizando los ganadores de las rondas previas.

In [41]:
# Mapeo estructurado de la fase final del torneo (Cuartos, Semis, Tercer Puesto y Final)
cuartos_fixtures = [
    {"partido_id": 97, "home_from": 89, "away_from": 90, "date": "2026-07-09", "stadium": "Gillette Stadium"},
    {"partido_id": 98, "home_from": 93, "away_from": 94, "date": "2026-07-10", "stadium": "SoFi Stadium"},
    {"partido_id": 99, "home_from": 91, "away_from": 92, "date": "2026-07-11", "stadium": "Hard Rock Stadium"},
    {"partido_id": 100, "home_from": 95, "away_from": 96, "date": "2026-07-11", "stadium": "Arrowhead Stadium"}
]

semifinales_fixtures = [
    {"partido_id": 101, "home_from": 97, "away_from": 98, "date": "2026-07-14", "stadium": "AT&T Stadium"},
    {"partido_id": 102, "home_from": 99, "away_from": 100, "date": "2026-07-15", "stadium": "Mercedes-Benz Stadium"}
]

finales_fixtures = [
    {"partido_id": 103, "home_from": 101, "away_from": 102, "type": "Tercer Puesto", "date": "2026-07-18", "stadium": "Hard Rock Stadium"},
    {"partido_id": 104, "home_from": 101, "away_from": 102, "type": "Final", "date": "2026-07-19", "stadium": "MetLife Stadium"}
]

print(f"✅ Estructura de fases finales cargada con éxito. Listo para simulación en cascada.")

✅ Estructura de fases finales cargada con éxito. Listo para simulación en cascada.


## 5. Construcción y Entrenamiento del Modelo Predictivo
Para predecir qué selección ganará cada cruce, utilizaremos un modelo de Machine Learning `RandomForestClassifier`. Lo entrenaremos simulando emparejamientos utilizando los datos de rendimiento que consolidamos previamente, evaluando variables como la efectividad histórica y los puntos por partido conseguidos en este Mundial 2026.

In [42]:
# 1. Crear un dataset de entrenamiento basado en diferencias de rendimiento de equipos históricos
X_train_list = []
y_train_list = []

for i, rowA in df_features.iterrows():
    for j, rowB in df_features.iterrows():
        if i != j:
            diff_efectividad = rowA['efectividad_historica_%'] - rowB['efectividad_historica_%']
            diff_forma = rowA['puntos_por_pj_2026'] - rowB['puntos_por_pj_2026']
            diff_dg = rowA['dg_2026'] - rowB['dg_2026']
            
            X_train_list.append([diff_efectividad, diff_forma, diff_dg])
            
            if rowA['puntos_por_pj_2026'] > rowB['puntos_por_pj_2026']:
                y_train_list.append(1)
            elif rowA['puntos_por_pj_2026'] < rowB['puntos_por_pj_2026']:
                y_train_list.append(0)
            else:
                y_train_list.append(1 if rowA['efectividad_historica_%'] > rowB['efectividad_historica_%'] else 0)

X = np.array(X_train_list)
y = np.array(y_train_list)

# 2. Inicializar y entrenar el modelo de Clasificación
model_mundial = RandomForestClassifier(n_estimators=150, random_state=42)
model_mundial.fit(X, y)

print("🤖 ¡Modelo RandomForestClassifier entrenado de forma impecable!")
print(f"Características analizadas: Jerarquía Histórica, Forma Reciente 2026 y Diferencia de Goles.")

🤖 ¡Modelo RandomForestClassifier entrenado de forma impecable!
Características analizadas: Jerarquía Histórica, Forma Reciente 2026 y Diferencia de Goles.


## 6. Ejecución del Simulador en Cascada hasta la Gran Final
Con el modelo entrenado, crearemos una función predictiva que resuelva las llaves pendientes de Dieciseisavos y arrastre los resultados automáticamente por Octavos, Cuartos, Semifinales y Final, imprimiendo en pantalla el desarrollo del torneo y proclamando al Campeón del Mundo 2026.

In [43]:
# Función experta para predecir el ganador entre dos selecciones
def predecir_ganador(equipo_a, equipo_b):
    try:
        feat_a = df_features[df_features['equipo'] == equipo_a].iloc[0]
        feat_b = df_features[df_features['equipo'] == equipo_b].iloc[0]
    except IndexError:
        return equipo_a

    diff_efectividad = feat_a['efectividad_historica_%'] - feat_b['efectividad_historica_%']
    diff_forma = feat_a['puntos_por_pj_2026'] - feat_b['puntos_por_pj_2026']
    diff_dg = feat_a['dg_2026'] - feat_b['dg_2026']
    
    features_partido = np.array([[diff_efectividad, diff_forma, diff_dg]])
    probabilidades = model_mundial.predict_proba(features_partido)[0]
    
    if probabilidades[1] >= 0.50:
        return equipo_a
    else:
        return equipo_b

# --- EJECUCIÓN DEL TORNEO EN CASCADA ---
resultados_partidos = {}

print("====== 🏆 SIMULACIÓN COMPLETA DE FASE ELIMINATORIA MUNDIAL 2026 🏆 ======\n")

# 1. Resolver Dieciseisavos de Final (73 al 88)
print("--- 📂 DIECISEISAVOS DE FINAL ---")
for _, row in df_cruces_32.iterrows():
    p_id = int(row['partido_id'])
    if row['played']:
        ganador = row['home_team'] if row['home_score'] > row['away_score'] else row['away_team']
        if 'ganador_penales' in row and pd.notna(row['ganador_penales']):
            ganador = row['ganador_penales']
    else:
        ganador = predecir_ganador(row['home_team'], row['away_team'])
        print(f"Partido {p_id}: {row['home_team']} vs {row['away_team']} ➡️ Predicción Clasifica: {ganador}")
    resultados_partidos[p_id] = ganador

# 2. Resolver Octavos de Final (89 al 96)
print("\n--- 🏟️ OCTAVOS DE FINAL ---")
for _, row in df_cruces_16.iterrows():
    p_id = int(row['partido_id'])
    home = resultados_partidos[int(row['home_team'].split()[-1])] if "Ganador" in row['home_team'] else row['home_team']
    away = resultados_partidos[int(row['away_team'].split()[-1])] if "Ganador" in row['away_team'] else row['away_team']
    
    ganador = predecir_ganador(home, away)
    resultados_partidos[p_id] = ganador
    print(f"Partido {p_id} ({row['date']}): {home} vs {away} ➡️ Clasifica: {ganador}")

# 3. Resolver Cuartos de Final (97 al 100)
print("\n--- 📉 CUARTOS DE FINAL ---")
for f in cuartos_fixtures:
    p_id = int(f['partido_id'])
    home = resultados_partidos[int(f['home_from'])]
    away = resultados_partidos[int(f['away_from'])]
    
    ganador = predecir_ganador(home, away)
    resultados_partidos[p_id] = ganador
    print(f"Partido {p_id} ({f['date']} en {f['stadium']}): {home} vs {away} ➡️ Clasifica: {ganador}")

# 4. Resolver Semifinales (101 y 102) -> ¡CORREGIDO AQUÍ SIN LLAMAR A STADIUM!
print("\n--- ⚡ SEMIFINALES ---")
for s in semifinales_fixtures:
    p_id = int(s['partido_id'])
    home = resultados_partidos[int(s['home_from'])]
    away = resultados_partidos[int(s['away_from'])]
    
    ganador = predecir_ganador(home, away)
    resultados_partidos[p_id] = ganador
    print(f"Partido {p_id} ({s['date']}): {home} vs {away} ➡️ Clasifica: {ganador}")

# 5. Tercer Puesto y Final (103 y 104)
print("\n--- 🏅 PARTIDOS DEFINITORIOS ---")
g_101 = resultados_partidos[101]
g_102 = resultados_partidos[102]

# Perdedores de Semifinales se calculan evaluando el rendimiento de los cruces de cuartos
p_101 = resultados_partidos[97] if g_101 == resultados_partidos[98] else resultados_partidos[98]
p_102 = resultados_partidos[99] if g_102 == resultados_partidos[100] else resultados_partidos[100]

tercer_lugar = predecir_ganador(p_101, p_102)
campeon = predecir_ganador(g_101, g_102)

print(f"🥉 Tercer Puesto (18-jul): {p_101} vs {p_102} ➡️ Ganador: {tercer_lugar}")
print(f"👑 ¡GRAN FINAL! (19-jul en MetLife): {g_101} vs {g_102} ➡️ ¡CAMPEÓN DEL MUNDO: {campeon}! 🎉")

====== 🏆 SIMULACIÓN COMPLETA DE FASE ELIMINATORIA MUNDIAL 2026 🏆 ======

--- 📂 DIECISEISAVOS DE FINAL ---
Partido 83: Portugal vs Croatia ➡️ Predicción Clasifica: Croatia
Partido 84: Spain vs Austria ➡️ Predicción Clasifica: Spain
Partido 85: Switzerland vs Algeria ➡️ Predicción Clasifica: Switzerland
Partido 86: Argentina vs Cape Verde ➡️ Predicción Clasifica: Argentina
Partido 87: Colombia vs Ghana ➡️ Predicción Clasifica: Colombia
Partido 88: Australia vs Egypt ➡️ Predicción Clasifica: Egypt

--- 🏟️ OCTAVOS DE FINAL ---
Partido 89 (2026-07-04): Paraguay vs France ➡️ Clasifica: France
Partido 90 (2026-07-04): Canada vs Morocco ➡️ Clasifica: Morocco
Partido 91 (2026-07-05): Brazil vs Norway ➡️ Clasifica: Brazil
Partido 92 (2026-07-05): Mexico vs England ➡️ Clasifica: Mexico
Partido 93 (2026-07-06): Croatia vs Spain ➡️ Clasifica: Spain
Partido 94 (2026-07-06): United States vs Belgium ➡️ Clasifica: United States
Partido 95 (2026-07-07): Argentina vs Egypt ➡️ Clasifica: Argentina
Partid